In [ ]:
#!/usr/bin/env python3
"""
compare_datasets.py

Compare genome-scale CD4+ T-cell Perturb-seq dataset (Marson 2025) with a
quiescence dataset to determine feasibility of using the Perturb-seq data
as external experimental ground truth for in-silico TF perturbation evaluation.

Usage:
    python3 compare_datasets.py [OPTIONS]

Options:
    --skip_download_quiescence   Skip downloading the quiescence dataset
    --skip_download_perturbseq   Skip downloading the Perturb-seq dataset
    --download_pseudobulk        Also download GWCD4i.pseudobulk_merged.h5ad
    --outdir OUTDIR              Output directory (default: outputs_comparison)
"""

import os
import sys
import json
import gzip
import shutil
import argparse
import subprocess
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# 0.  Argument parsing
# ─────────────────────────────────────────────────────────────
parser = argparse.ArgumentParser(description="Compare Perturb-seq vs Quiescence datasets")
parser.add_argument("--skip_download_quiescence", action="store_true",
                    help="Skip downloading the quiescence dataset")
parser.add_argument("--skip_download_perturbseq", action="store_true",
                    help="Skip downloading the Perturb-seq dataset")
parser.add_argument("--download_pseudobulk", action="store_true",
                    help="Also download GWCD4i.pseudobulk_merged.h5ad from S3")
parser.add_argument("--outdir", default="outputs_comparison",
                    help="Output directory (default: outputs_comparison)")
args = parser.parse_args()

OUTDIR = args.outdir
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs("./my_data_folder", exist_ok=True)
os.makedirs("./perturbseq_data", exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 0b. Dependency checks
# ─────────────────────────────────────────────────────────────
def check_import(pkg, pip_name=None):
    import importlib
    try:
        return importlib.import_module(pkg)
    except ImportError:
        pip_name = pip_name or pkg
        print(f"[ERROR] Missing package '{pkg}'. Install with:")
        print(f"        pip install {pip_name}")
        sys.exit(1)

requests  = check_import("requests")
np        = check_import("numpy")
pd        = check_import("pandas")
sc        = check_import("scanpy", "scanpy")
ad        = check_import("anndata")

print("=" * 70)
print("CD4+ T-CELL DATASET COMPATIBILITY ANALYSIS")
print("Marson 2025 Perturb-seq  ↔  Quiescence dataset")
print("=" * 70)

# ─────────────────────────────────────────────────────────────
# 1a. Download quiescence dataset (Dropbox)
# ─────────────────────────────────────────────────────────────
DROPBOX_URL = (
    "https://www.dropbox.com/scl/fi/e38rl88uk3m5ux4v29vdt/"
    "compressed_cleaned_labelled_t_cell_data.gz"
    "?rlkey=e0ww30defqgh8dhyak28ig8lr&e=1&dl=1"
)
COMPRESSED_FILENAME = "downloaded_data.gz"
EXTRACT_FOLDER      = "./my_data_folder"
TARGET_FILENAME     = "mydata.h5ad"
H5AD_PATH           = os.path.join(EXTRACT_FOLDER, TARGET_FILENAME)

if not args.skip_download_quiescence:
    print("\n" + "─" * 70)
    print("STEP 1a — Download quiescence dataset from Dropbox")
    print("─" * 70)

    # Download .gz
    if os.path.exists(COMPRESSED_FILENAME):
        gz_size = os.path.getsize(COMPRESSED_FILENAME) / (1024**2)
        print(f"  Compressed file already exists ({gz_size:.1f} MB) — skipping download.")
        print(f"  Delete '{COMPRESSED_FILENAME}' to re-download.")
    else:
        print(f"  Downloading from Dropbox…")
        try:
            with requests.get(DROPBOX_URL, stream=True, timeout=300) as r:
                r.raise_for_status()
                total_size    = int(r.headers.get("content-length", 0))
                total_mb      = total_size / (1024**2)
                downloaded    = 0
                print(f"  Total size: {total_mb:.1f} MB")
                with open(COMPRESSED_FILENAME, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)
                            if total_size > 0:
                                pct = downloaded / total_size * 100
                                print(f"  {downloaded/(1024**2):.1f} / {total_mb:.1f} MB ({pct:.1f}%)", end="\r")
            print(f"\n  ✓ Downloaded: {COMPRESSED_FILENAME}")
        except Exception as e:
            print(f"  ✗ Download failed: {e}")
            print(f"  Try: wget '{DROPBOX_URL}' -O {COMPRESSED_FILENAME}")
            sys.exit(1)

    # Validate gzip
    print("  Validating gzip…")
    try:
        with gzip.open(COMPRESSED_FILENAME, "rb") as f:
            f.read(1024)
        gz_size = os.path.getsize(COMPRESSED_FILENAME) / (1024**2)
        print(f"  ✓ Valid gzip ({gz_size:.1f} MB)")
    except Exception as e:
        print(f"  ✗ Corrupted .gz file: {e}")
        os.remove(COMPRESSED_FILENAME)
        print("  Deleted corrupted file. Re-run to re-download.")
        sys.exit(1)

    # Extract
    if os.path.exists(H5AD_PATH):
        print(f"  Removing existing {H5AD_PATH}")
        os.remove(H5AD_PATH)

    print("  Extracting .gz → .h5ad …")
    try:
        with gzip.open(COMPRESSED_FILENAME, "rb") as f_in, open(H5AD_PATH, "wb") as f_out:
            extracted = 0
            while True:
                chunk = f_in.read(10 * 1024 * 1024)
                if not chunk:
                    break
                f_out.write(chunk)
                extracted += len(chunk)
                print(f"  Extracted: {extracted/(1024**2):.1f} MB", end="\r")
        print(f"\n  ✓ Extracted to: {H5AD_PATH}")
    except Exception as e:
        print(f"  ✗ Extraction failed: {e}")
        if os.path.exists(H5AD_PATH):
            os.remove(H5AD_PATH)
        sys.exit(1)

    # Verify
    h5ad_size = os.path.getsize(H5AD_PATH) / (1024**3)
    print(f"  Extracted file size: {h5ad_size:.2f} GB")
    try:
        tmp = sc.read_h5ad(H5AD_PATH, backed="r")
        print(f"  ✓ Valid AnnData: {tmp.n_obs:,} cells × {tmp.n_vars:,} genes")
        del tmp
    except Exception as e:
        print(f"  ✗ Validation failed: {e}")
        sys.exit(1)
else:
    print("\n[SKIP] Quiescence dataset download skipped (--skip_download_quiescence).")
    if not os.path.exists(H5AD_PATH):
        print(f"[ERROR] Expected file not found: {H5AD_PATH}")
        sys.exit(1)

# ─────────────────────────────────────────────────────────────
# 1b. Download Perturb-seq dataset from S3
# ─────────────────────────────────────────────────────────────
S3_BUCKET   = "s3://genome-scale-tcell-perturb-seq/marson2025_data"
PS_DE_FILE  = "GWCD4i.DE_stats.h5ad"
PS_PB_FILE  = "GWCD4i.pseudobulk_merged.h5ad"
PS_DE_PATH  = f"./perturbseq_data/{PS_DE_FILE}"
PS_PB_PATH  = f"./perturbseq_data/{PS_PB_FILE}"

def s3_download(s3_uri, local_path, label):
    """Download from S3 using aws CLI (unsigned) or boto3 fallback."""
    if os.path.exists(local_path):
        size_mb = os.path.getsize(local_path) / (1024**2)
        print(f"  {label} already exists ({size_mb:.1f} MB) — skipping.")
        return

    # Try aws cli
    aws_ok = shutil.which("aws") is not None
    if aws_ok:
        print(f"  Downloading {label} via aws s3 cp …")
        cmd = ["aws", "s3", "cp", "--no-sign-request", s3_uri, local_path]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f"  ✓ Downloaded: {local_path}")
            return
        else:
            print(f"  aws CLI failed: {result.stderr.strip()}")
            print("  Falling back to boto3…")

    # boto3 fallback
    try:
        import boto3
        from botocore import UNSIGNED
        from botocore.config import Config
    except ImportError:
        print("  [ERROR] boto3 not installed. Install with: pip install boto3")
        sys.exit(1)

    # Parse s3://bucket/key
    s3_path   = s3_uri.replace("s3://", "")
    bucket    = s3_path.split("/")[0]
    key       = "/".join(s3_path.split("/")[1:])

    print(f"  Downloading {label} via boto3 (unsigned)…")
    try:
        s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
        meta = s3.head_object(Bucket=bucket, Key=key)
        total_bytes = meta["ContentLength"]
        total_mb    = total_bytes / (1024**2)
        print(f"  File size: {total_mb:.1f} MB")
        downloaded_bytes = 0

        def progress(bytes_transferred):
            nonlocal downloaded_bytes
            downloaded_bytes += bytes_transferred
            pct = downloaded_bytes / total_bytes * 100
            print(f"  {downloaded_bytes/(1024**2):.1f} / {total_mb:.1f} MB ({pct:.1f}%)", end="\r")

        s3.download_file(bucket, key, local_path, Callback=progress)
        print(f"\n  ✓ Downloaded: {local_path}")
    except Exception as e:
        print(f"  ✗ boto3 download failed: {e}")
        sys.exit(1)

if not args.skip_download_perturbseq:
    print("\n" + "─" * 70)
    print("STEP 1b — Download Perturb-seq dataset from S3")
    print("─" * 70)
    s3_download(f"{S3_BUCKET}/{PS_DE_FILE}", PS_DE_PATH, PS_DE_FILE)
    if args.download_pseudobulk:
        s3_download(f"{S3_BUCKET}/{PS_PB_FILE}", PS_PB_PATH, PS_PB_FILE)
else:
    print("\n[SKIP] Perturb-seq download skipped (--skip_download_perturbseq).")
    if not os.path.exists(PS_DE_PATH):
        print(f"[ERROR] Expected file not found: {PS_DE_PATH}")
        sys.exit(1)

# ─────────────────────────────────────────────────────────────
# 2. Load both datasets
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("STEP 2 — Load datasets")
print("─" * 70)

print(f"  Loading quiescence dataset: {H5AD_PATH}")
qdata = sc.read_h5ad(H5AD_PATH)
print(f"  ✓ Quiescence: {qdata.n_obs:,} cells × {qdata.n_vars:,} genes")

print(f"  Loading Perturb-seq DE stats: {PS_DE_PATH}")
psdata = sc.read_h5ad(PS_DE_PATH)
print(f"  ✓ Perturb-seq DE stats: {psdata.n_obs:,} rows × {psdata.n_vars:,} genes")
print(f"    (rows = perturbation-condition pairs; columns = genes)")

# ─────────────────────────────────────────────────────────────
# 3. Determine gene identifier types and normalize
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("STEP 3 — Gene identifier detection and normalization")
print("─" * 70)

def detect_id_type(var_names_sample):
    """Detect gene ID type from a sample of var_names."""
    sample = list(var_names_sample[:20])
    n_ensg = sum(1 for g in sample if str(g).startswith("ENSG"))
    n_ensg_versioned = sum(1 for g in sample if str(g).startswith("ENSG") and "." in str(g))
    if n_ensg > len(sample) * 0.8:
        if n_ensg_versioned > len(sample) * 0.5:
            return "ensembl_versioned"
        return "ensembl"
    return "symbol"

def strip_ensembl_version(series):
    """Strip version suffix from Ensembl IDs (ENSG00000123.4 → ENSG00000123)."""
    return series.str.replace(r"\.\d+$", "", regex=True)

def normalize_var_names(adata, label):
    """
    Normalize var_names in place.
    Returns (adata, id_type, had_version).
    Also looks for a 'gene_name'/'gene_symbols' column as fallback.
    """
    id_type    = detect_id_type(adata.var_names)
    had_version = False

    print(f"  [{label}] Detected gene ID type: {id_type}")
    print(f"  [{label}] Sample var_names: {list(adata.var_names[:5])}")

    if id_type == "ensembl_versioned":
        adata.var_names = strip_ensembl_version(pd.Index(adata.var_names))
        had_version = True
        print(f"  [{label}] Stripped version suffixes. New sample: {list(adata.var_names[:5])}")
        id_type = "ensembl"

    # Check for auxiliary gene_name column
    symbol_cols = [c for c in adata.var.columns if c.lower() in
                   ("gene_name", "gene_names", "gene_symbol", "gene_symbols",
                    "symbol", "hgnc_symbol", "feature_name")]
    if symbol_cols:
        print(f"  [{label}] Found symbol column(s): {symbol_cols}")
        adata.var["_symbol"] = adata.var[symbol_cols[0]].astype(str)
    else:
        if id_type == "symbol":
            adata.var["_symbol"] = adata.var_names.astype(str)
        else:
            adata.var["_symbol"] = ""

    # Deduplicate var_names (keep first occurrence)
    if adata.var_names.duplicated().any():
        n_dup = adata.var_names.duplicated().sum()
        print(f"  [{label}] WARNING: {n_dup} duplicate var_names — keeping first occurrence.")
        adata = adata[:, ~adata.var_names.duplicated()].copy()

    return adata, id_type, had_version

qdata,  q_id_type,  q_had_ver  = normalize_var_names(qdata,  "Quiescence")
psdata, ps_id_type, ps_had_ver = normalize_var_names(psdata, "Perturb-seq")

# ─────────────────────────────────────────────────────────────
# 4. Compute gene overlap
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("STEP 4 — Gene overlap computation")
print("─" * 70)

q_genes  = set(qdata.var_names)
ps_genes = set(psdata.var_names)

shared_genes = q_genes & ps_genes
q_only       = q_genes - ps_genes
ps_only      = ps_genes - q_genes
gene_union   = q_genes | ps_genes

pct_of_q  = len(shared_genes) / len(q_genes)  * 100 if q_genes  else 0
pct_of_ps = len(shared_genes) / len(ps_genes) * 100 if ps_genes else 0

print(f"  Quiescence genes      : {len(q_genes):>7,}  (ID type: {q_id_type})")
print(f"  Perturb-seq genes     : {len(ps_genes):>7,}  (ID type: {ps_id_type})")
print(f"  Shared genes          : {len(shared_genes):>7,}")
print(f"  Union genes           : {len(gene_union):>7,}")
print(f"  % overlap (of quies.) : {pct_of_q:.1f}%")
print(f"  % overlap (of ps.)    : {pct_of_ps:.1f}%")
print(f"  Quiescence-only genes : {len(q_only):>7,}")
print(f"  Perturb-seq-only genes: {len(ps_only):>7,}")

if len(shared_genes) < 5000:
    print(f"\n  ⚠  WARNING: Only {len(shared_genes):,} shared genes — below recommended 5 000 threshold!")
else:
    print(f"\n  ✓ Gene overlap is adequate (≥5 000 genes).")

# If ID types differ, attempt symbol-based rescue
if q_id_type != ps_id_type:
    print(f"\n  ID types differ ({q_id_type} vs {ps_id_type}). Attempting symbol-based rescue…")
    q_sym  = set(qdata.var["_symbol"].dropna()) - {"", "nan"}
    ps_sym = set(psdata.var["_symbol"].dropna()) - {"", "nan"}
    sym_shared = q_sym & ps_sym
    print(f"  Symbol-based shared genes: {len(sym_shared):,}")
    if len(sym_shared) > len(shared_genes):
        print("  → Symbol matching yields more overlap; using symbols for downstream steps.")
        shared_genes = sym_shared
        # Remap index to symbol for overlap purposes
        q_genes  = q_sym
        ps_genes = ps_sym

# Write CSVs
pd.DataFrame({"gene": sorted(shared_genes)}).to_csv(
    f"{OUTDIR}/shared_genes.csv", index=False)
pd.DataFrame({"gene": sorted(q_only)}).to_csv(
    f"{OUTDIR}/quiescence_only_genes.csv", index=False)
pd.DataFrame({"gene": sorted(ps_only)}).to_csv(
    f"{OUTDIR}/perturbseq_only_genes.csv", index=False)
print(f"\n  Wrote: {OUTDIR}/shared_genes.csv  ({len(shared_genes):,} entries)")
print(f"  Wrote: {OUTDIR}/quiescence_only_genes.csv  ({len(q_only):,} entries)")
print(f"  Wrote: {OUTDIR}/perturbseq_only_genes.csv  ({len(ps_only):,} entries)")

# Gene category breakdown
def classify_genes(gene_set):
    mt  = sum(1 for g in gene_set if str(g).upper().startswith("MT-"))
    rp  = sum(1 for g in gene_set if str(g).upper().startswith(("RPL","RPS","MRPL","MRPS")))
    nc  = sum(1 for g in gene_set if any(str(g).upper().startswith(p)
              for p in ("LINC","MIR","SNORD","SNORA","SCARNA","RMRP","RNU")))
    return {"mt": mt, "ribosomal": rp, "noncoding": nc, "other": len(gene_set) - mt - rp - nc}

q_cats  = classify_genes(q_genes)
ps_cats = classify_genes(ps_genes)
sh_cats = classify_genes(shared_genes)
print("\n  Gene category breakdown:")
print(f"  {'Category':<15} {'Quiescence':>12} {'Perturb-seq':>12} {'Shared':>8}")
print(f"  {'─'*15} {'─'*12} {'─'*12} {'─'*8}")
for cat in ("mt", "ribosomal", "noncoding", "other"):
    print(f"  {cat:<15} {q_cats[cat]:>12,} {ps_cats[cat]:>12,} {sh_cats[cat]:>8,}")

# ─────────────────────────────────────────────────────────────
# 5. TF overlap and candidate TF list
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("STEP 5 — TF overlap and candidate list extraction")
print("─" * 70)

# Identify perturbation metadata columns
obs_cols = list(psdata.obs.columns)
print(f"  Perturb-seq .obs columns ({len(obs_cols)}): {obs_cols[:20]}")

# Find the column that names the perturbed gene / TF
tf_col_candidates = [c for c in obs_cols if any(kw in c.lower() for kw in
    ("gene_name", "target", "perturbation", "guide", "tf", "contrast_gene"))]
print(f"  TF/target column candidates: {tf_col_candidates}")

# Pick best column
tf_col = None
for candidate in ("target_contrast_gene_name", "gene_name", "target_gene",
                   "perturbation", "guide_target"):
    if candidate in obs_cols:
        tf_col = candidate
        break
if tf_col is None and tf_col_candidates:
    tf_col = tf_col_candidates[0]

if tf_col is None:
    print("  ⚠  No TF/perturbation column found — trying all string columns for gene-like values.")
    # Last resort: find a column with ENSG or gene-symbol-like values
    for col in obs_cols:
        vals = psdata.obs[col].astype(str)
        n_genelike = sum(1 for v in vals[:50] if v.startswith("ENSG") or
                         (v.isidentifier() and len(v) <= 12))
        if n_genelike > 20:
            tf_col = col
            print(f"  Using column '{tf_col}' as TF identifier.")
            break

if tf_col is None:
    print("  ✗ Cannot identify TF column. Skipping TF analysis.")
    tf_records = []
else:
    print(f"  Using column '{tf_col}' as TF identifier.")

    # Condition column
    cond_col = None
    for c in ("condition", "stim", "stimulation", "treatment", "timepoint", "sample"):
        if c in obs_cols:
            cond_col = c
            break
    if cond_col is None:
        cond_candidates = [c for c in obs_cols if any(kw in c.lower()
                           for kw in ("cond","stim","treat","time"))]
        cond_col = cond_candidates[0] if cond_candidates else None

    print(f"  Condition column: {cond_col}")
    if cond_col:
        print(f"  Unique conditions: {sorted(psdata.obs[cond_col].unique())}")

    # Significance / off-target columns
    ontarget_col = "ontarget_significant"  if "ontarget_significant"  in obs_cols else None
    offtarget_col = "offtarget_flag"       if "offtarget_flag"        in obs_cols else None

    print(f"  ontarget_significant col: {ontarget_col}")
    print(f"  offtarget_flag col      : {offtarget_col}")

    # Build candidate TF table
    obs_df = psdata.obs.copy()

    if ontarget_col:
        # Handle boolean stored as string
        mask = obs_df[ontarget_col].astype(str).str.lower().isin(["true","1","yes","t"])
        obs_df = obs_df[mask]
        print(f"  Rows after ontarget_significant filter: {len(obs_df):,}")

    if offtarget_col:
        mask_off = ~obs_df[offtarget_col].astype(str).str.lower().isin(["true","1","yes","t"])
        obs_df = obs_df[mask_off]
        print(f"  Rows after offtarget_flag exclusion:     {len(obs_df):,}")

    # Normalize TF names (strip version suffix if Ensembl)
    obs_df[tf_col] = obs_df[tf_col].astype(str).str.replace(r"\.\d+$", "", regex=True)

    # Intersect with genes present in quiescence dataset
    obs_df["_in_quiescence"] = obs_df[tf_col].isin(q_genes)
    obs_in_q = obs_df[obs_df["_in_quiescence"]]
    print(f"  TF perturbations with TF present in quiescence dataset: {len(obs_in_q):,}")

    # Aggregate per TF × condition
    effect_col = next((c for c in obs_cols if "effect" in c.lower() or "lfc" in c.lower()
                       or "log" in c.lower()), None)
    ncells_col  = next((c for c in obs_cols if "ncell" in c.lower() or "n_cell" in c.lower()
                        or "cell_count" in c.lower()), None)
    ndepth_col  = next((c for c in obs_cols if "downstream" in c.lower()
                        or "n_de" in c.lower() or "n_sig" in c.lower()), None)

    group_cols = [tf_col]
    if cond_col:
        group_cols.append(cond_col)

    agg_dict = {}
    if effect_col:
        agg_dict["ontarget_effect_size"] = (effect_col, "mean")
    if ncells_col:
        agg_dict["n_cells_target"] = (ncells_col, "sum")
    if ndepth_col:
        agg_dict["n_downstream"] = (ndepth_col, "mean")

    if agg_dict:
        tf_records_df = (obs_in_q.groupby(group_cols)
                         .agg(**agg_dict)
                         .reset_index()
                         .rename(columns={tf_col: "TF",
                                          cond_col: "condition"} if cond_col else {tf_col: "TF"}))
    else:
        tf_records_df = (obs_in_q[group_cols].drop_duplicates()
                         .rename(columns={tf_col: "TF",
                                          cond_col: "condition"} if cond_col else {tf_col: "TF"}))

    tf_records_df.to_csv(f"{OUTDIR}/candidate_TFs_by_condition.csv", index=False)
    print(f"  Wrote: {OUTDIR}/candidate_TFs_by_condition.csv  ({len(tf_records_df):,} rows)")

    # Summary counts
    tf_records = tf_records_df.to_dict("records")

    if "condition" in tf_records_df.columns:
        per_cond = tf_records_df.groupby("condition")["TF"].nunique().to_dict()
    else:
        per_cond = {"all": tf_records_df["TF"].nunique()}

    total_tf = tf_records_df["TF"].nunique() if "TF" in tf_records_df.columns else 0
    print(f"  Total unique candidate TFs (in quiescence): {total_tf}")
    for cond, n in per_cond.items():
        print(f"    {cond}: {n} TFs")

# ─────────────────────────────────────────────────────────────
# 6. Biological context alignment
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("STEP 6 — Biological context alignment")
print("─" * 70)

# Activation/quiescence marker sets (human T-cell canonical)
QUIESCENT_MARKERS  = ["IL7R", "LTB", "CCR7", "TCF7", "LEF1", "SELL", "KLF2", "S1PR1"]
ACTIVATED_MARKERS  = ["IL2", "IFNG", "TNFRSF9", "CD69", "JUNB", "FOS", "MYC",
                       "CXCR3", "GZMB", "PRF1"]

def find_obs_label_col(adata, keywords):
    """Find obs column most likely to encode cell state / condition labels."""
    for col in adata.obs.columns:
        if any(kw in col.lower() for kw in keywords):
            return col
    # Fallback: search column values
    for col in adata.obs.columns:
        vals = adata.obs[col].astype(str).str.lower()
        if vals.str.contains("|".join(keywords)).any():
            return col
    return None

q_label_col = find_obs_label_col(qdata,
    ["quiesc","rest","activ","stim","state","cell_type","condition","label","cluster"])

print(f"  Quiescence obs label column: {q_label_col}")
if q_label_col:
    print(f"  Unique values: {sorted(qdata.obs[q_label_col].unique())[:20]}")

context_lines = []
context_lines.append("=" * 60)
context_lines.append("BIOLOGICAL CONTEXT ALIGNMENT REPORT")
context_lines.append("=" * 60)
context_lines.append("")

def compute_module_score(adata, markers, score_name):
    """Compute scanpy module score for a marker list that is present in dataset."""
    present = [g for g in markers if g in adata.var_names]
    if len(present) < 2:
        return None, present
    try:
        sc.tl.score_genes(adata, gene_list=present, score_name=score_name, use_raw=False)
        return score_name, present
    except Exception as e:
        print(f"    score_genes failed ({e}); using manual mean.")
        idx = [list(adata.var_names).index(g) for g in present]
        import scipy.sparse as sp
        X = adata.X
        if sp.issparse(X):
            X = X.toarray()
        scores = X[:, idx].mean(axis=1)
        adata.obs[score_name] = scores
        return score_name, present

# Score quiescence dataset
print("  Computing module scores in quiescence dataset…")
q_act_col,  q_act_present  = compute_module_score(qdata, ACTIVATED_MARKERS,  "score_activated")
q_rest_col, q_rest_present = compute_module_score(qdata, QUIESCENT_MARKERS, "score_quiescent")

context_lines.append("--- QUIESCENCE DATASET ---")
context_lines.append(f"Label column used : {q_label_col}")
context_lines.append(f"Quiescent markers found ({len(q_rest_present)}/{len(QUIESCENT_MARKERS)}): {q_rest_present}")
context_lines.append(f"Activation markers found ({len(q_act_present)}/{len(ACTIVATED_MARKERS)}): {q_act_present}")

if q_label_col and q_act_col and q_rest_col:
    q_summary = qdata.obs.groupby(q_label_col)[[q_act_col, q_rest_col]].mean()
    context_lines.append("\nMean module scores per label:")
    context_lines.append(q_summary.to_string())
    context_lines.append("")

# Score perturb-seq dataset (if we can identify a condition column & expression data)
print("  Computing module scores in Perturb-seq dataset…")
ps_act_col,  ps_act_present  = compute_module_score(psdata, ACTIVATED_MARKERS,  "score_activated")
ps_rest_col, ps_rest_present = compute_module_score(psdata, QUIESCENT_MARKERS, "score_quiescent")

ps_cond_col_score = cond_col if 'cond_col' in dir() and cond_col else None

context_lines.append("--- PERTURB-SEQ DATASET ---")
context_lines.append(f"Condition column  : {ps_cond_col_score}")
context_lines.append(f"Quiescent markers found ({len(ps_rest_present)}/{len(QUIESCENT_MARKERS)}): {ps_rest_present}")
context_lines.append(f"Activation markers found ({len(ps_act_present)}/{len(ACTIVATED_MARKERS)}): {ps_act_present}")

if ps_cond_col_score and ps_act_col and ps_rest_col:
    ps_summary = psdata.obs.groupby(ps_cond_col_score)[[ps_act_col, ps_rest_col]].mean()
    context_lines.append("\nMean module scores per condition:")
    context_lines.append(ps_summary.to_string())
    context_lines.append("")
    context_lines.append("Interpretation guide:")
    context_lines.append("  • High quiescent score ↔ low activation  → 'Rest' condition")
    context_lines.append("  • High activation score ↔ low quiescent  → 'Stim48hr' condition")
    context_lines.append("  • Intermediate scores                     → 'Stim8hr' condition")
    context_lines.append("")
    context_lines.append("Recommended alignment:")
    context_lines.append("  Quiescence dataset  'quiescent'  ↔  Perturb-seq 'Rest'")
    context_lines.append("  Quiescence dataset  'activated'  ↔  Perturb-seq 'Stim48hr' (best)")
    context_lines.append("  Quiescence dataset  'activated'  ↔  Perturb-seq 'Stim8hr'  (early)")
else:
    context_lines.append("\n[WARNING] Could not compute per-condition module scores.")
    context_lines.append("  Manual inspection required.")
    context_lines.append("")
    context_lines.append("Expected alignment based on biology:")
    context_lines.append("  Quiescent cells  ↔  Perturb-seq 'Rest'    (unstimulated CD4+)")
    context_lines.append("  Activated cells  ↔  Perturb-seq 'Stim48hr' (anti-CD3/CD28, 48 h)")
    context_lines.append("  Early activation ↔  Perturb-seq 'Stim8hr'  (anti-CD3/CD28, 8 h)")

context_text = "\n".join(context_lines)
with open(f"{OUTDIR}/context_alignment_report.txt", "w") as fh:
    fh.write(context_text)
print(f"  Wrote: {OUTDIR}/context_alignment_report.txt")

# ─────────────────────────────────────────────────────────────
# Summary counts JSON
# ─────────────────────────────────────────────────────────────
summary = {
    "quiescence": {
        "n_cells": int(qdata.n_obs),
        "n_genes": int(qdata.n_vars),
        "id_type": q_id_type,
        "had_version_suffix": q_had_ver,
    },
    "perturbseq": {
        "n_rows": int(psdata.n_obs),
        "n_genes": int(psdata.n_vars),
        "id_type": ps_id_type,
        "had_version_suffix": ps_had_ver,
    },
    "gene_overlap": {
        "shared": int(len(shared_genes)),
        "quiescence_only": int(len(q_only)),
        "perturbseq_only": int(len(ps_only)),
        "union": int(len(gene_union)),
        "pct_of_quiescence": round(pct_of_q, 2),
        "pct_of_perturbseq": round(pct_of_ps, 2),
        "overlap_sufficient_5k": bool(len(shared_genes) >= 5000),
    },
    "tf_analysis": {
        "total_candidate_tfs_in_quiescence": int(total_tf) if 'total_tf' in dir() else None,
        "per_condition": {k: int(v) for k, v in per_cond.items()} if 'per_cond' in dir() else {},
    },
    "gene_categories_shared": {k: int(v) for k, v in sh_cats.items()},
}
with open(f"{OUTDIR}/summary_counts.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print(f"\n  Wrote: {OUTDIR}/summary_counts.json")

# ─────────────────────────────────────────────────────────────
# 7. Feasibility verdict
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 70)
print("STEP 7 — Writing feasibility report")
print("─" * 70)

overlap_ok  = len(shared_genes) >= 5000
total_tf_n  = total_tf if 'total_tf' in dir() else 0
tfs_ok      = total_tf_n >= 50
bio_ok      = True  # CD4+ T-cell in both datasets — biologically coherent by design
knockdown_ok = True  # Marson 2025 uses CRISPRi (knockdown, not activation)

verdict = "✅ YES — feasible as external ground truth" if (overlap_ok and tfs_ok and bio_ok) \
          else "⚠  CONDITIONAL — see risks below"

md_lines = [
    "# Feasibility Report: Perturb-seq as Ground Truth for TF Perturbation Evaluation",
    "",
    f"**Generated automatically by compare_datasets.py**",
    "",
    "---",
    "",
    "## 1. Dataset Structural Comparison",
    "",
    "| Property | Quiescence | Perturb-seq DE stats |",
    "|---|---|---|",
    f"| Cells / rows | {qdata.n_obs:,} | {psdata.n_obs:,} |",
    f"| Genes | {qdata.n_vars:,} | {psdata.n_vars:,} |",
    f"| Gene ID type | {q_id_type} | {ps_id_type} |",
    f"| Version suffix present | {q_had_ver} | {ps_had_ver} |",
    "",
    "---",
    "",
    "## 2. Gene Overlap Analysis",
    "",
    "| Metric | Value |",
    "|---|---|",
    f"| Shared genes | {len(shared_genes):,} |",
    f"| Quiescence-only genes | {len(q_only):,} |",
    f"| Perturb-seq-only genes | {len(ps_only):,} |",
    f"| Gene union | {len(gene_union):,} |",
    f"| % overlap (of quiescence) | {pct_of_q:.1f}% |",
    f"| % overlap (of perturb-seq) | {pct_of_ps:.1f}% |",
    f"| Overlap ≥ 5,000 genes? | {'✅ Yes' if overlap_ok else '❌ No'} |",
    "",
    "### Gene Category Breakdown (Shared Genes)",
    "",
    "| Category | Quiescence | Perturb-seq | Shared |",
    "|---|---|---|---|",
] + [
    f"| {cat} | {q_cats[cat]:,} | {ps_cats[cat]:,} | {sh_cats[cat]:,} |"
    for cat in ("mt", "ribosomal", "noncoding", "other")
] + [
    "",
    "---",
    "",
    "## 3. Biological Compatibility Assessment",
    "",
    "Both datasets are **human primary CD4+ T cells**.",
    "",
    "| Axis | Quiescence dataset | Perturb-seq |",
    "|---|---|---|",
    "| Species | Human | Human |",
    "| Cell type | CD4+ T cells | CD4+ T cells (GWCD4i) |",
    "| Perturbation type | In silico (model) | CRISPRi knockdown ✅ |",
    "| Resting state | Quiescent cells | 'Rest' condition |",
    "| Activated state | Activated cells | 'Stim8hr' / 'Stim48hr' |",
    "| Stimulation protocol | Unknown (see obs) | Anti-CD3/CD28 ± IL2 |",
    "",
    "**Recommended condition mapping:**",
    "",
    "- Quiescence dataset **quiescent** cells  →  Perturb-seq **Rest**",
    "- Quiescence dataset **activated** cells  →  Perturb-seq **Stim48hr** (primary)",
    "- Quiescence dataset **early activated** →  Perturb-seq **Stim8hr** (secondary)",
    "",
    "---",
    "",
    "## 4. TF Perturbation Overlap",
    "",
    f"| Metric | Value |",
    "|---|---|",
    f"| Total candidate TFs (present in quiescence dataset) | {total_tf_n} |",
]

if 'per_cond' in dir():
    for cond, n in per_cond.items():
        md_lines.append(f"| TFs in condition: {cond} | {n} |")

md_lines += [
    "",
    "---",
    "",
    "## 5. Ground Truth Extraction Validation",
    "",
    "For each TF *t* and condition *s*, the ground truth perturbation vector is:",
    "",
    "```",
    "ΔGT(t, s, g) = log_fold_change(TF-knockdown vs control)",
    "```",
    "",
    "Extracted from `GWCD4i.DE_stats.h5ad` obs rows filtered to:",
    "- `ontarget_significant == True`",
    "- `offtarget_flag != True` (if column present)",
    "",
    "The `.X` matrix of this AnnData stores per-gene DE statistics",
    "(log-fold-changes or z-scores) — one row per perturbation-condition pair.",
    "",
    "---",
    "",
    "## 6. Feasibility Verdict",
    "",
    f"### Overall: {verdict}",
    "",
    f"| Question | Answer |",
    "|---|---|",
    f"| Gene overlap ≥ 5,000? | {'✅ Yes' if overlap_ok else '❌ No — insufficient overlap'} |",
    f"| Biological alignment defensible? | {'✅ Yes' if bio_ok else '❌ No'} |",
    f"| ≥ 50 TF perturbations available? | {'✅ Yes' if tfs_ok else f'⚠  Only {total_tf_n} TFs found'} |",
    "| CRISPRi knockdown only? | ✅ Yes — knockdown, not overexpression |",
    "| Ground truth valid for single-TF only? | ✅ Yes — combinatorial not directly covered |",
    "",
    "---",
    "",
    "## 7. Risks and Mitigation Strategies",
    "",
    "| Risk | Severity | Mitigation |",
    "|---|---|---|",
    "| Different donors / genetic background | Medium | Use shared TF perturbation signatures; average over conditions |",
    "| Activation protocol may differ | Medium | Validate marker scores; use Stim48hr as primary comparator |",
    "| 'Rest' ≠ true quiescence (short rest from culture) | Medium | Report separately; check KLF2/S1PR1/TCF7 expression |",
    "| Gene ID mismatch (Ensembl vs symbol) | Low–Medium | Strip version suffixes; apply symbol fallback mapping |",
    "| Off-target CRISPRi effects | Low | Filter to `ontarget_significant==True`; exclude `offtarget_flag==True` |",
    "| TF regulon differences across donors | Low–Medium | Use rank-based metrics (AUROC, rank overlap) rather than absolute LFC |",
    "| No combinatorial ground truth | High (for combos) | Explicitly exclude combinatorial predictions from evaluation |",
    "| Batch effects (different platforms) | Medium | Normalize both datasets; use relative metrics |",
    "",
    "### Required Preprocessing Steps",
    "",
    "1. Strip Ensembl version suffixes in both datasets.",
    "2. Restrict to shared gene set for all comparisons.",
    "3. Filter perturb-seq to `ontarget_significant == True`.",
    "4. Match biological conditions (quiescent → Rest; activated → Stim48hr).",
    "5. Normalize expression (log1p CPM or scran normalization) before scoring.",
    "6. Use rank-based evaluation metrics (AUROC, Spearman, rank overlap) to reduce batch sensitivity.",
    "",
    "### Fatal Incompatibilities",
    "",
    "**None identified**, assuming gene overlap ≥ 5,000.",
    "If gene overlap < 5,000, the evaluation will be severely underpowered.",
    "",
    "---",
    "",
    "## 8. Unresolved Uncertainties",
    "",
    "- Exact donor count and HLA diversity in quiescence dataset (not extracted).",
    "- Whether quiescence dataset uses CRISPRi, pharmacological, or serum-starvation quiescence.",
    "- Whether the 'Rest' condition in perturb-seq uses matched resting duration.",
    "- TF baseline expression differences — may affect knockdown efficiency comparison.",
    "- Whether TF regulons are stable across activation states (affects which condition to use).",
    "",
    "---",
    "",
    "*End of report.*",
]

feasibility_text = "\n".join(md_lines)
with open(f"{OUTDIR}/feasibility_report.md", "w") as fh:
    fh.write(feasibility_text)
print(f"  Wrote: {OUTDIR}/feasibility_report.md")

# ─────────────────────────────────────────────────────────────
# Final summary
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)
print(f"\n  Verdict: {verdict}")
print(f"\n  Output files in: ./{OUTDIR}/")
output_files = [
    "shared_genes.csv",
    "quiescence_only_genes.csv",
    "perturbseq_only_genes.csv",
    "candidate_TFs_by_condition.csv",
    "summary_counts.json",
    "context_alignment_report.txt",
    "feasibility_report.md",
]
for f in output_files:
    path = f"{OUTDIR}/{f}"
    exists = "✓" if os.path.exists(path) else "✗ MISSING"
    print(f"    {exists}  {path}")
print()

usage: colab_kernel_launcher.py [-h] [--skip_download_quiescence]
                                [--skip_download_perturbseq]
                                [--download_pseudobulk] [--outdir OUTDIR]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-3a8c67ba-b5dd-40ad-822d-ea00fa261621.json


SystemExit: 2